In [ ]:
#Import Necessary Packages/Modules
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt # Run "pip install matplotlib" if it doesn't work
from scipy import stats
from joblib import Parallel, delayed
from skeLCS import eLCS   # pip install scikit-eLCS
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB  
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report)
from sklearn.preprocessing import label_binarize

#Reusable constants.  You can change these to match your own data and preferences.
RAW_FILE   = "medical_insurance.csv"
CLEAN_FILE = "medical_insurance_cleaned_v2.csv"   # output of Task 3
SEED       = 42
N_SPLITS   = 5
N_REPEATS  = 3      # 15 paired scores per model; same folds as the Task 5 notebook
N_FEATURES = 4
N_JOBS     = -1     # set to 1 if you hit memory/pickling problems

In [2]:
LEAKAGE_COLS = ["is_high_risk", "annual_premium", "monthly_premium", "annual_medical_cost",
                "claims_count", "avg_claim_amount", "total_claims_paid"]

raw   = pd.read_csv(RAW_FILE,   keep_default_na=False, na_values=[""])
clean = pd.read_csv(CLEAN_FILE, keep_default_na=False, na_values=[""])
assert raw["person_id"].is_unique

# Minimal processing only (as in Task 2): drop leakage/ID columns, text -> integer codes
Xr_df = raw.drop(columns=LEAKAGE_COLS + ["risk_score", "person_id"])
for c in Xr_df.select_dtypes(exclude="number"):
    Xr_df[c] = Xr_df[c].astype("category").cat.codes
feature_names = list(Xr_df.columns)
Xr = Xr_df.values.astype(float)

# Labels: tertile edges from the CLEANED data, applied to every row (raw rows included)
q = clean["risk_score"].quantile([1/3, 2/3]).values
edges = [-np.inf, q[0], q[1], np.inf]
yr = pd.cut(raw["risk_score"], edges, labels=False).values
BAND_NAMES = ["Low", "Medium", "High"]

# Locate each cleaned row in the raw file, so both views share rows, features and labels
pos = raw.reset_index().set_index("person_id").loc[clean["person_id"], "index"].values
Xc, yc = Xr[pos], yr[pos]
assert np.allclose(clean["age"].values, raw["age"].values[pos]) and np.allclose(clean["bmi"].values, raw["bmi"].values[pos])
assert (yc == pd.qcut(clean["risk_score"], 3, labels=False).values).all()

print(f"raw: {Xr.shape} | cleaned: {Xc.shape} | removed by cleaning: {len(raw) - len(clean)}")
print("class shares (cleaned):", np.round(np.bincount(yc) / len(yc), 3))

raw: (100000, 45) | cleaned: (96713, 45) | removed by cleaning: 3287
class shares (cleaned): [0.349 0.328 0.322]


In [3]:
# Wrapper for scikit-learn compatibility
class ScikitELCSWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, N=1000, learning_iterations=10000, do_correct_set_subsumption=False, random_state=None):
        self.N = N
        self.learning_iterations = learning_iterations
        self.do_correct_set_subsumption = do_correct_set_subsumption
        self.random_state = random_state

    def fit(self, X, y):
        y_int = y.astype(int)
        self.model_ = eLCS(N=self.N, learning_iterations=self.learning_iterations,
                           do_correct_set_subsumption=self.do_correct_set_subsumption,
                           random_state=self.random_state)
        self.model_.fit(X, y_int)
        self.classes_ = np.unique(y_int)
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

def selector():
    return SelectFromModel(DecisionTreeClassifier(max_depth=5, random_state=SEED),
                           max_features=N_FEATURES, threshold=-np.inf)

DT  = lambda: DecisionTreeClassifier(max_depth=5, random_state=SEED)
RF  = lambda: RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=1)
LR  = lambda: LogisticRegression(max_iter=1000)
NB  = lambda: GaussianNB()

# name -> (model, training data: "raw" or "clean", feature set description)
models = {
    "eLCS original, raw data":          (Pipeline([("elcs", ScikitELCSWrapper(random_state=SEED))]), "raw",   "45, uncleaned"),
    "eLCS original, preprocessed":      (Pipeline([("elcs", ScikitELCSWrapper(random_state=SEED))]), "clean", "45"),
    "eLCS improved":                    (Pipeline([("select", selector()), ("scale", MinMaxScaler()),
                                          ("elcs", ScikitELCSWrapper(random_state=SEED, do_correct_set_subsumption=True,
                                                                     learning_iterations=20000, N=1500))]), "clean", "4 selected"),
    "Decision tree":                    (DT(),                                                  "clean", "45"),
    "Random forest":                    (RF(),                                                  "clean", "45"),
    "Logistic regression":              (Pipeline([("s", StandardScaler()), ("m", LR())]),      "clean", "45"),
    "Gaussian naive Bayes":             (NB(),                                                  "clean", "45"),
    "Decision tree (4 selected)":       (Pipeline([("sel", selector()), ("m", DT())]),          "clean", "4 selected"),
    "Random forest (4 selected)":       (Pipeline([("sel", selector()), ("m", RF())]),          "clean", "4 selected"),
    "Logistic regression (4 selected)": (Pipeline([("sel", selector()), ("s", StandardScaler()), ("m", LR())]), "clean", "4 selected"),
}
REF = "eLCS improved"

NameError: name 'GaussianNB' is not defined